# Fiorell.IA LoRA — Master Runbook (Colab / Azure / AWS)

Conservative wrapper for the existing training → export → eval → decision workflow.

This notebook does **not** change:
- base model (`Qwen/Qwen2.5-3B-Instruct`);
- LoRA method;
- dataset semantics;
- abstention/refusal behavior;
- evaluation harness logic.

It only centralizes path configuration, preflight checks and artifact validation.


In [ ]:
# 00_config — edit only this cell
from pathlib import Path

RUN_ENV = "colab"  # colab | azure | aws | local

REPO_ROOT = Path("/content/regulatory-insight-engine") if RUN_ENV == "colab" else Path.cwd()
CONFIG_PATH = REPO_ROOT / "fiorellia/training/configs/config_lora_behavior_20260421.yaml"

TRAIN_SCRIPT = REPO_ROOT / "fiorellia/training/train_lora_behavior_v1.py"

ARTIFACT_ROOT = Path("/content/drive/MyDrive/fiorellia/artifacts") if RUN_ENV == "colab" else REPO_ROOT / "artifacts/fiorellia"
ADAPTER_ZIP = ARTIFACT_ROOT / "fiorellia_lora_adapter.zip"
METRICS_JSON = ARTIFACT_ROOT / "metrics_summary.json"
FINAL_VERDICT_MD = ARTIFACT_ROOT / "final_verdict.md"

SYSTEM_PROMPT_PATH = REPO_ROOT / "fiorellia/prompts/system_prompt.md"
EVAL_SET_PATH = REPO_ROOT / "fiorellia/eval/eval_set.jsonl"
BASELINE_JSONL_PATH = REPO_ROOT / "fiorellia/eval/baseline.jsonl"

print("RUN_ENV:", RUN_ENV)
print("REPO_ROOT:", REPO_ROOT)
print("ARTIFACT_ROOT:", ARTIFACT_ROOT)

## 01_dependencies

Run once on a clean runtime. Avoid scattered `pip install -U` cells later in the notebook.

In [ ]:
# Colab/Azure/AWS conservative dependency cell.
# Keep ranges broad enough for Colab T4, narrow enough to avoid common breakage.
%pip install -q \
  "transformers>=4.45,<4.52" \
  "datasets>=2.20,<3.0" \
  "accelerate>=0.33,<1.0" \
  "peft>=0.12,<0.16" \
  "trl>=0.9,<0.13" \
  "bitsandbytes>=0.43,<0.46" \
  "safetensors>=0.4" \
  "pyyaml>=6.0" 

In [ ]:
# 02_preflight
import sys
sys.path.insert(0, str(REPO_ROOT))

from fiorellia.training.fiorellia_colab_pipeline import (
    check_cuda,
    load_config,
    validate_config,
    require_file,
)

gpu_info = check_cuda(require_gpu=True)
config = load_config(CONFIG_PATH)
resolved = validate_config(config, REPO_ROOT)

ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

print("GPU:", gpu_info)
print("Dataset:", resolved["dataset_path"])
print("Output dir:", resolved["output_dir"])
print("Artifact root:", ARTIFACT_ROOT)

## 03_training

This cell delegates to the existing training script. It does not change training semantics.

In [ ]:
# 03_training — keep disabled until preflight is green
import subprocess, sys

require_file(TRAIN_SCRIPT, "training script")
cmd = [sys.executable, str(TRAIN_SCRIPT), "--config", str(CONFIG_PATH)]
print("+", " ".join(cmd))
subprocess.run(cmd, cwd=str(REPO_ROOT), check=True)

print("Training completed")

In [ ]:
# 04_export — validate adapter directory and create zip
from fiorellia.training.fiorellia_colab_pipeline import zip_adapter, validate_adapter_zip

adapter_dir = resolved["output_dir"]
created_zip = zip_adapter(adapter_dir, ADAPTER_ZIP)
validate_adapter_zip(created_zip)

print("Adapter zip:", created_zip)

## 05_eval

Eval remains owned by the existing eval notebook/script. This runbook validates required inputs before you run the current eval logic.

In [ ]:
# 05_eval_preflight — run before executing existing eval cells/script
from fiorellia.training.fiorellia_colab_pipeline import validate_adapter_zip

validate_adapter_zip(ADAPTER_ZIP)
require_file(SYSTEM_PROMPT_PATH, "system prompt")
require_file(EVAL_SET_PATH, "eval set")
require_file(BASELINE_JSONL_PATH, "baseline JSONL")

print("Eval inputs validated")
print("Adapter zip:", ADAPTER_ZIP)
print("System prompt:", SYSTEM_PROMPT_PATH)
print("Eval set:", EVAL_SET_PATH)
print("Baseline:", BASELINE_JSONL_PATH)

In [ ]:
# 06_compare_and_finalize
# Fill metrics from the existing eval harness output, then write standard artifacts.
from fiorellia.training.fiorellia_colab_pipeline import write_json, write_final_verdict

metrics = {
    "inscopegrounded": None,
    "unsupportedabstention": None,
    "outofscoperefusal": None,
    "priority_cases_ok": None,
}

thresholds = {
    "inscopegrounded": 0.80,
    "unsupportedabstention": 0.90,
    "outofscoperefusal": 0.95,
}

def metric_ok(name):
    value = metrics.get(name)
    return value is not None and value >= thresholds[name]

go = (
    metric_ok("inscopegrounded")
    and metric_ok("unsupportedabstention")
    and metric_ok("outofscoperefusal")
    and metrics.get("priority_cases_ok") is True
)

verdict = "GO" if go else "NO-GO"
write_json(metrics, METRICS_JSON)
write_final_verdict(FINAL_VERDICT_MD, verdict, metrics)

print("CONCLUSIONE DEFINITIVA")
print("Esito run:", verdict)
print("Metrics:", metrics)
print("Metrics JSON:", METRICS_JSON)
print("Final verdict:", FINAL_VERDICT_MD)